# Calibration suite v2 — a walkthrough on real PYTHIA data

This notebook works through the post-review update to this repo
([`docs/PLAN_UPDATES.md`](../docs/PLAN_UPDATES.md), work packages WP1–WP4) on the
**real PYTHIA 8.3 sample** in `cpp/test_data/jets.root` — 54 007 groomed jets written by
`cpp/apps/pythia_driver.cpp` — rather than the synthetic generator the older notebooks use.

The question it answers is narrow and load-bearing:

> **Given a trained hadron→parton posterior, how do we know it is calibrated —
> and how do we know our calibration test isn't rigged?**

The short version of why the suite had to grow. The previous test was the
simulation-based-calibration rank of the **multiplicity** `n`. That is a real test for
`ar_junipr_v2`, whose length model is an implicit per-step continue/stop product. But
`ar_junipr_v3` trains `q(N|x)` by **direct NLL on N** — so SBC-on-N certifies the very
marginal the model optimizes. It passes near-tautologically, and a v2-vs-v3 comparison
judged on it is biased toward v3 *by construction*.

So we add three diagnostics that test what SBC-on-N cannot, and demonstrate each against
a **deliberately broken control**, so you can see what failure looks like and not just
what success looks like:

| § | What | Tests |
|---|---|---|
| 4 | the tautology, shown | why SBC-on-N cannot referee v2 vs v3 |
| 5 | **per-coordinate PITs** | the *kinematics*, coordinate by coordinate |
| 6 | **region stratification** | whether calibration holds *locally* on the Lund plane |
| 7 | **TARP** | the *whole tree*, jointly, in the physics metric |
| 8 | `exact_likelihood` | which families' `log_prob` is a density at all (WP1) |
| 9 | cross-attention | the fixed-length bottleneck (WP3) |

Runs top to bottom on CPU or Apple MPS in roughly 15 minutes.

In [ ]:
import math, sys, time, warnings
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import gridspec

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
warnings.filterwarnings("ignore", category=UserWarning)

from h2p_rsd_junipr.config import load_config
from h2p_rsd_junipr.data.datamodule import LundDataModule
from h2p_rsd_junipr.data.rntuple import load_rntuple
from h2p_rsd_junipr.data.stats import check_multiplicity_support, multiplicity_stats
from h2p_rsd_junipr.eval.calibration import (
    REGION_LABELS, coordinate_pits, run_calibration, run_tarp,
)
from h2p_rsd_junipr.geometry import Geometry
from h2p_rsd_junipr.train.logging import CSVJSONLLogger
from h2p_rsd_junipr.train.trainer import Trainer, build_components, seed_everything, select_device

JETS_ROOT = REPO / "cpp" / "test_data" / "jets.root"
RUN_ROOT = REPO / "runs" / "calibration_v2_walkthrough"
EPOCHS = 12          # ~20 s/epoch on MPS for the AR families
K_DRAWS = 100        # posterior draws per jet in the calibration loops
N_EVAL = 300         # held-out jets per diagnostic

# --- plotting: a validated categorical palette, assigned in fixed order --------
# Slots 1-4 of the reference palette; adjacent-pair CVD deltaE 9.2, normal-vision 27.6.
C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
C_GOOD, C_CRIT = "#0ca30c", "#d03b3b"          # status colours, never reused as series
INK, INK2, GRID = "#0b0b0b", "#52514e", "#dcdbd6"
SEQ = "Blues"                                   # single hue, light -> dark

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "axes.titlesize": 10,
    "axes.labelsize": 9, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "legend.frameon": False, "figure.dpi": 120, "lines.linewidth": 2.0,
})

def finish(ax, title=None, xlabel=None, ylabel=None):
    """Recessive frame: only the axes we read, grid behind the marks."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    if title: ax.set_title(title, color=INK)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    return ax

device = select_device()
seed_everything(0)
print(f"torch {torch.__version__} on {device}   |   data: {JETS_ROOT.relative_to(REPO)}")

---
## 1. The data — real PYTHIA, not the synthetic generator

`cpp/test_data/jets.root` is a ROOT **RNTuple** written by the C++ generation stage. Each
row is one jet and carries its own grooming provenance, so the file is self-describing:
we never have to remember which `z_cut` produced it.

The two sequences per jet are **node-unaligned by design** — `x_*` is the hadron-level
primary Lund sequence, `y_*` the parton-level one, and there is no per-node
correspondence between them. That is the whole reason this is a *posterior* problem
rather than a regression.

In [ ]:
jets = load_rntuple(str(JETS_ROOT), "Jets")
j0 = jets[0]
stats = multiplicity_stats(jets)

print(f"generator      : {j0['generator']}")
print(f"grooming       : z_cut={j0['z_cut']:g}  beta={j0['beta']:g}  kt_floor={j0['kt_floor']:g} GeV")
print(f"jets           : {stats['n_jets']:,}   events: {len({j['event'] for j in jets}):,}")
print(f"parton mult N  : mean {stats['mean']:.2f}   max {stats['max']}   P(N=0) {stats['frac_empty']:.3f}")
print(f"hadron mult    : mean {np.mean([len(j['x'][0]) for j in jets]):.2f}")

In [ ]:
geom = Geometry()   # default (0,6) x (0,6), 10 x 10 cells
fig = plt.figure(figsize=(10.5, 3.3))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.32)

# (a,b) the two Lund densities, each its own single-hue sequential ramp
for k, (key, label) in enumerate((("x", "hadron level $x$"), ("y", "parton level $y$"))):
    ax = fig.add_subplot(gs[0, k])
    u = np.concatenate([j[key][0] for j in jets]); v = np.concatenate([j[key][1] for j in jets])
    h = ax.hist2d(u, v, bins=[np.linspace(0, 6, 61), np.linspace(0, 6, 61)],
                  cmap=SEQ, cmin=1)
    fig.colorbar(h[3], ax=ax, label="emissions", pad=0.02)
    finish(ax, f"Lund plane, {label}", r"$\ln 1/\Delta R$", r"$\ln k_t$")
    ax.grid(False)

# (c) the multiplicity marginals -- the quantity the OLD calibration test used
ax = fig.add_subplot(gs[0, 2])
bins = np.arange(-0.5, 9.5)
ax.hist([len(j["y"][0]) for j in jets], bins=bins, color=C_BLUE, alpha=0.85,
        label="parton $N$", rwidth=0.9)
ax.hist([len(j["x"][0]) for j in jets], bins=bins, histtype="step", color=C_ORANGE,
        label="hadron $n_x$", linewidth=2)
ax.legend(loc="upper right")
finish(ax, "multiplicity marginals", "primary splittings", "jets")
plt.show()

print("Hadronization ADDS emissions (orange sits right of blue) and smears their kinematics —")
print("inverting that smearing is the task; the multiplicity shift is only its most visible part.")

---
## 2. WP4 — the multiplicity-support guard

`ar_junipr_v3`, `cinn`, `diffusion` and `cfm` model the length with a **categorical**
`q(N|x)` over `N = 0..model.max_emissions`. That support is finite, and the v2
continue/stop head's was not.

A truth sequence past the support is **clamped into the last bin**. It then receives the
wrong likelihood — for every such jet, silently, with no signature whatsoever in the loss
curve. Grooming parameters move that tail (a looser `z_cut` or a lower `k_t` floor admits
more primary emissions), so the check has to run against the data actually loaded.

`train` now does exactly that, and hard-errors above `P_data(N > max_emissions) = 1e-3`.

In [ ]:
cfg_v3 = load_config([
    "model=ar_junipr_v3", "encoder=gru",
    "data=rntuple", f"data.path={JETS_ROOT}",
    f"trainer.max_epochs={EPOCHS}", "trainer.batch_size=128",
])
ok = check_multiplicity_support(jets, cfg_v3)
print(f"OK  : max N = {ok['max']}  vs  support {ok['support']}  ->  tail fraction {ok['tail_fraction']:.1e}\n")

# ...and what it looks like when the support is genuinely too small for the data:
try:
    check_multiplicity_support(jets, load_config(["model=ar_junipr_v3", "model.max_emissions=3"]))
except ValueError as exc:
    print("FIRES:", str(exc)[:430], "...")

Note what the message carries: the offending `z_cut` / `β` / `k_t` floor **and** the
concrete bound to raise `max_emissions` to. A guard that only says "something is wrong"
costs the reader the debugging session it was supposed to save.

This sample is comfortably inside the default support (max `N` = 6 against 25), so it is
not a live risk here — but it would be at a looser grooming, and that is exactly when
nobody would think to check.

---
## 3. Train the posterior on the real data

`ar_junipr_v3`: a bi-GRU encoder over the hadron sequence, a GRU decoder over the parton
tree, a categorical `q(N|x)` multiplicity head, a categorical cell head, and a continuous
within-cell coordinate head (truncated normals for `du, dv`, a normal for `ln z`, a von
Mises for `ψ`).

That last head is what §5 tests — and note that **none of its four coordinates is the
multiplicity**, which is the entire point.

In [ ]:
def train(cfg, tag):
    """Train one model and return it with its best val NLL. Reuses a cached run if present."""
    seed_everything(0)
    g = Geometry.from_config(cfg.geometry)
    dm = LundDataModule(cfg, g).setup()
    run_dir = RUN_ROOT / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    model, opt, sched = build_components(cfg, g, device)
    n_par = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    logger = CSVJSONLLogger(run_dir, tensorboard=False)
    tr = Trainer(model, opt, sched, dm.loaders(), cfg, logger, device, run_dir, dm.fingerprint)
    best = tr.fit()
    logger.close()
    print(f"[{tag}] {n_par/1e3:.1f}k params | best val NLL/jet = {best:.3f} | {time.time()-t0:.0f}s")
    return tr.model.eval(), best, dm, g

model, nll_v3, dm, geom = train(cfg_v3, "ar_junipr_v3")
train_ds, val_ds = dm.datasets()
print(f"held-out jets available for the diagnostics: {len(val_ds):,}")

In [ ]:
hist = np.genfromtxt(RUN_ROOT / "ar_junipr_v3" / "metrics.csv", delimiter=",", names=True)
fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.plot(hist["epoch"], hist["train_nll"], color=C_BLUE, label="train")
ax.plot(hist["epoch"], hist["val_nll"], color=C_ORANGE, label="validation")
ax.annotate(f"{nll_v3:.2f}", xy=(hist["epoch"][-1], nll_v3), xytext=(-4, 8),
            textcoords="offset points", ha="right", color=INK2, fontsize=8)
ax.legend()
finish(ax, "training curve (real PYTHIA jets)", "epoch", "NLL / jet [nats]")
plt.show()

---
## 4. The problem: SBC-on-N cannot referee this model

Simulation-based calibration (Talts et al., arXiv:1804.06788) ranks the true value among
posterior draws; for a calibrated posterior the rank is uniform. Applied to the
multiplicity `n`, on `ar_junipr_v3`, it looks excellent.

It has to. `q(N|x)` was fit by maximum likelihood **directly on N**, and the draws come
from that same head. The test is asking the model to reproduce a one-dimensional marginal
it was explicitly trained to reproduce — and saying nothing at all about the 4-dimensional
kinematics per emission, which is where essentially all of the model's capacity lives.

The suite below therefore reports SBC-N as before (it is still the right test for `v2`)
but **gates the v2-vs-v3 A/B on the diagnostics that follow instead**.

In [ ]:
t0 = time.time()
base_metrics = run_calibration(model, val_ds, geom, device, K=K_DRAWS, n_jets=N_EVAL)
print(f"\n[{time.time()-t0:.0f}s]")

Read that as: **the length marginal is fine, and we already knew it would be.** Nothing
above constrains the coordinate heads.

---
## 5. WP2.1 — per-coordinate PITs

The probability-integral transform: if `Y ~ q` and `F` is `q`'s CDF, then `F(Y) ~ U(0,1)`.
So evaluate each coordinate's **conditional CDF at the truth**, teacher-forced, and
histogram the result. Under a calibrated head it is flat.

For the AR heads this transform is exact and lives in *physical* coordinates — the
truncated-normal CDF for the within-cell offsets `du, dv` (built from the very same
normalizer the likelihood divides by, so the PIT and the likelihood cannot drift apart),
the normal CDF for `ln z`, and a von Mises CDF for `ψ`. That last one needed a new helper:
`distributions.vonmises_cdf`, a Fourier series whose Bessel ratios come from a
continued-fraction recurrence — overflow-free at any `κ` and elementwise only, so it still
runs on MPS.

**Read the shape, not only the number.** A flat histogram is calibrated. A **U** — mass
piled at 0 *and* 1 — means the truth keeps landing in the tails, i.e. the head is
**over-confident**, too narrow. A **dome** means over-dispersed. The KS distance is the
scalar summary; its 95% critical value is `1.36/√n`, drawn on each panel.

In [ ]:
pits = coordinate_pits(model, val_ds, geom, device, n_jets=N_EVAL,
                       stratify_regions=True, verbose=True)

In [ ]:
def pit_panel(ax, entry, title, color=C_BLUE):
    h = np.asarray(entry["hist"], float); edges = np.asarray(entry["edges"], float)
    ctr, w = 0.5 * (edges[:-1] + edges[1:]), edges[1] - edges[0]
    ax.bar(ctr, h, width=w * 0.9, color=color, zorder=3)
    ax.axhline(h.sum() / len(h), color=INK2, ls="--", lw=1.4, zorder=4)
    crit = 1.36 / math.sqrt(max(entry["n"], 1))
    verdict = "calibrated" if entry["ks"] < crit else "MISCALIBRATED"
    vcol = C_GOOD if entry["ks"] < crit else C_CRIT
    ax.set_title(title, color=INK)
    ax.text(0.5, 1.14, f"KS {entry['ks']:.3f} (crit {crit:.3f}) — {verdict}",
            transform=ax.transAxes, ha="center", fontsize=8, color=vcol)
    ax.set_xlim(0, 1); ax.set_ylim(0, max(h.max() * 1.35, 1))
    return finish(ax, None, "PIT", None)

names = list(pits["names"])
fig, axes = plt.subplots(1, len(names), figsize=(3.0 * len(names), 3.0))
for ax, nm in zip(np.atleast_1d(axes), names):
    pit_panel(ax, pits["coords"][nm], {"du": r"$\delta u$  (within-cell $\ln 1/\Delta R$)",
                                       "dv": r"$\delta v$  (within-cell $\ln k_t$)",
                                       "ln_z": r"$\ln z$", "psi": r"$\psi$"}[nm])
np.atleast_1d(axes)[0].set_ylabel("emissions")
fig.suptitle("per-coordinate PIT — flat = calibrated, U = over-confident, dome = over-dispersed",
             y=1.13, fontsize=10)
plt.show()

### 5a. What a *broken* head looks like

The panels above are only interpretable if we know what failure looks like on the same
axes. So: halve the predicted widths — a model that is exactly twice as confident as it
should be — and re-run the identical diagnostic on the identical data.

In [ ]:
from contextlib import contextmanager

@contextmanager
def scaled_widths(m, scale):
    """Temporarily rescale the coordinate head's widths — a controlled miscalibration.

    Patches `_coord_params` on the instance rather than wrapping the model: every
    consumer (`coordinate_cdfs`, `log_prob`, `describe_sequence`) reaches the head
    through that one method, so patching it is the only way a wrapper's override is
    guaranteed to be seen. Only the WIDTHS move — means, cells and q(N|x) are
    untouched, so any change in the PIT is attributable to confidence alone."""
    orig, s = m._coord_params, float(scale)
    def patched(coord_in):
        du_m, dv_m, du_s, dv_s, lz_m, lz_s, mu, kap = orig(coord_in)
        return (du_m, dv_m, du_s * s, dv_s * s, lz_m, lz_s * s, mu, kap / s**2)
    m._coord_params = patched
    try:
        yield m
    finally:
        del m._coord_params          # restore the class method

fig, axes = plt.subplots(2, len(names), figsize=(3.0 * len(names), 5.8))
for row, (scale, label, colour) in enumerate([(0.5, r"$\sigma \times 0.5$  (over-confident)", C_ORANGE),
                                              (2.0, r"$\sigma \times 2$  (over-dispersed)", C_VIOLET)]):
    with scaled_widths(model, scale) as m_broken:
        broken = coordinate_pits(m_broken, val_ds, geom, device, n_jets=N_EVAL, verbose=False)
    for ax, nm in zip(axes[row], names):
        pit_panel(ax, broken["coords"][nm], nm, color=colour)
    axes[row][0].set_ylabel(label, fontsize=9)
fig.suptitle("controls: the same jets, judged by a deliberately mis-scaled head", y=1.02, fontsize=10)
fig.tight_layout()
plt.show()

print("Top row: mass at both edges — the truth lands in the tails because the head is too narrow.")
print("Bottom row: mass in the middle — the head is too broad and the truth is always 'typical'.")
print("Both are unmistakable, and neither is visible to SBC-on-N.")

### 5b. Is the *late* emission calibrated, or only the first?

The AR decoder is teacher-forced during training and free-running at sampling time, so
errors compound along the sequence — classic exposure bias. If it is present, PIT quality
degrades with the emission index. The report carries that breakdown for free.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.3))
palette = [C_BLUE, C_ORANGE, C_AQUA, C_VIOLET]
for nm, colour in zip(names, palette):
    by_i = pits["coords"][nm]["by_emission_index"]
    idx = sorted(by_i, key=int)
    ks = [by_i[i]["ks"] for i in idx]
    ax.plot([int(i) for i in idx], ks, marker="o", ms=5, color=colour, label=nm)
    ax.annotate(nm, xy=(int(idx[-1]), ks[-1]), xytext=(5, 0), textcoords="offset points",
                color=INK2, fontsize=8, va="center")
first = pits["coords"][names[0]]["by_emission_index"]
n_first = first[sorted(first, key=int)[0]]["n"]
ax.axhline(1.36 / math.sqrt(n_first), color=INK2, ls="--", lw=1.2)
ax.text(0.02, 1.36 / math.sqrt(n_first), " 95% critical value (1st emission)", color=INK2,
        fontsize=8, va="bottom", transform=ax.get_yaxis_transform())
ax.set_xlim(-0.3, len(idx) - 0.2)
finish(ax, "PIT KS by emission index — a rise means exposure bias", "emission index $t$",
       "KS distance to $U(0,1)$")
plt.show()

print("Later emissions have fewer jets behind them, so the critical value rises too —")
print("compare each point against its own n, not against the first-emission line alone.")

---
## 6. WP2.2 — region stratification: *where* does calibration hold?

Every number so far is an average over the whole Lund plane. A posterior can be perfectly
calibrated on average and badly wrong in one corner — and the corners are exactly where
the physics lives: the collinear/soft region is hadronization-dominated, the wide-angle
hard region is perturbative.

So bin every metric by the **Lund quadrant of the leading (hardest) emission**. This is
the direct precondition for any *localized* claim — a heavy-ion measurement that quotes
modification in one region needs calibration demonstrated in that region, not on average.

In [ ]:
metrics = run_calibration(model, val_ds, geom, device, K=K_DRAWS, n_jets=N_EVAL,
                          stratify_regions=True, pit_coords=True, verbose=False)
by_region = metrics["by_region"]

fig = plt.figure(figsize=(10.2, 3.4))
gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1.05, 1.25, 1.25], wspace=0.38)

# (a) what the four quadrants ARE, on the plane itself
ax = fig.add_subplot(gs[0, 0])
mid_u, mid_v = 3.0, 3.0
ax.axvline(mid_u, color=INK2, lw=1.2); ax.axhline(mid_v, color=INK2, lw=1.2)
for (x, y, lab) in [(1.5, 1.5, "wide\nsoft"), (1.5, 4.5, "wide\nhard"),
                    (4.5, 1.5, "narrow\nsoft"), (4.5, 4.5, "narrow\nhard")]:
    ax.text(x, y, lab, ha="center", va="center", color=INK, fontsize=9)
ax.set_xlim(0, 6); ax.set_ylim(0, 6); ax.grid(False)
finish(ax, "the four strata", r"$\ln 1/\Delta R$", r"$\ln k_t$")

# (b) coverage per region, against its 0.68 target
ax = fig.add_subplot(gs[0, 1])
labs = [r for r in REGION_LABELS if r in by_region]
cov = [by_region[r]["coverage_68"] for r in labs]
n_j = [by_region[r]["n_jets"] for r in labs]
bars = ax.bar(np.arange(len(labs)), cov, width=0.62, color=C_BLUE, zorder=3)
ax.axhline(0.68, color=C_CRIT, ls="--", lw=1.5, zorder=4)
ax.text(len(labs) - 0.5, 0.68, " target 0.68", color=C_CRIT, fontsize=8, va="bottom", ha="right")
for b, c, n in zip(bars, cov, n_j):
    ax.annotate(f"{c:.2f}\nn={n}", (b.get_x() + b.get_width() / 2, c), xytext=(0, 4),
                textcoords="offset points", ha="center", fontsize=7.5, color=INK2)
ax.set_xticks(np.arange(len(labs))); ax.set_xticklabels(labs, rotation=18)
ax.set_ylim(0, 1.05)
finish(ax, "68% coverage of the true leading cell", None, "coverage")

# (c) the same strata, judged by the coordinate PIT instead
ax = fig.add_subplot(gs[0, 2])
width = 0.2
for k, (nm, colour) in enumerate(zip(names, palette)):
    reg = pits["coords"][nm].get("by_region", {})
    vals = [reg.get(r, {}).get("ks", np.nan) for r in labs]
    ax.bar(np.arange(len(labs)) + (k - 1.5) * width, vals, width=width * 0.92,
           color=colour, label=nm, zorder=3)
ax.set_xticks(np.arange(len(labs))); ax.set_xticklabels(labs, rotation=18)
ax.legend(ncol=2)
finish(ax, "PIT KS per region", None, "KS distance")
plt.show()

for r in labs:
    e = by_region[r]
    print(f"  {r:>12}  n={e['n_jets']:>4}   cov68 {e['coverage_68']:.2f}   "
          f"SBC chi2 {e['sbc_chi2_uniform']:6.1f}   rank mean {e['sbc_rank_mean']:.3f}")

Regions are unequally populated — the soft/collinear corner holds most jets — so a region
with few jets has correspondingly loose error bars, and its `n` is printed on the bar for
exactly that reason. What you are looking for is not four identical numbers but the
absence of a *systematic* gap between the perturbative and hadronization-dominated corners.

---
## 7. WP2.3 — TARP: the whole tree, jointly, in the physics metric

Both diagnostics so far are **marginal**: SBC on `N`, PITs on one coordinate at a time. A
posterior can have every marginal calibrated and still get the *joint* structure wrong —
correlations between multiplicity and kinematics, for instance.

TARP (Lemos et al., ICML 2023, arXiv:2302.03026) tests the joint distribution directly and
needs only a distance. Ours is the **perturbative-Lund EMD** already in the repo for the
MBR point estimator (Komiske–Metodiev–Thaler, arXiv:1902.02346): each tree becomes a
`k_t`-weighted point cloud on the Lund plane, compared by optimal transport. So the test
runs in the metric the physics actually cares about, not in an arbitrary embedding.

Per jet: draw a reference tree `r`, and ask what fraction of posterior draws are closer to
`r` than the truth is:

$$f = \tfrac{1}{K}\Big[\#\{k: d(r,y_k) < d(r,y_\text{true})\} + \tfrac12 \#\{k: d(r,y_k) = d(r,y_\text{true})\}\Big]$$

Under a calibrated posterior each `f` is uniform, so the expected-coverage probability
`ECP(α) = P(f < α)` equals `α` — the diagonal. (The half-weight on ties is the same
mid-rank convention the SBC statistic uses; without it the discrete cell chains tie often
enough to fake over-dispersion.)

**The sign is the diagnosis**: below the diagonal ⇒ over-confident, above ⇒ over-dispersed.

In [ ]:
class SharpPosterior:
    """The trained model sampled at a LOW temperature — a controlled over-confidence.

    `cont_temperature < 1` sharpens the cell logits at sampling time only; the trained
    likelihood is untouched. `run_tarp` reaches the model through `sample_batch` alone,
    so overriding that one method is enough — and it is exactly the failure TARP should
    catch: a posterior that concentrates on a handful of trees."""
    def __init__(self, base, temperature):
        self.base, self.t = base, float(temperature)
    def __getattr__(self, k):
        return getattr(self.base, k)
    def sample_batch(self, xf, nx, n, max_emissions=25):
        return self.base.sample(xf, nx, n, max_emissions=max_emissions, cont_temperature=self.t)

t0 = time.time()
tarp = run_tarp(model, val_ds, geom, device, K=K_DRAWS, n_jets=N_EVAL, n_refs=100, seed=0)
tarp_sharp = run_tarp(SharpPosterior(model, 0.05), val_ds, geom, device,
                      K=K_DRAWS, n_jets=N_EVAL, n_refs=100, seed=0, verbose=False)
print(f"\n[{time.time()-t0:.0f}s]")

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.4))
a = np.asarray(tarp["alpha"]); ax.plot([0, 1], [0, 1], color=INK2, ls="--", lw=1.3)
ax.text(0.62, 0.55, "calibrated", color=INK2, fontsize=8, rotation=38)
for res, colour, lab in ((tarp, C_BLUE, "trained model"),
                         (tarp_sharp, C_ORANGE, "control: temperature 0.05")):
    e = np.asarray(res["ecp"])
    ax.plot(a, e, color=colour, marker="o", ms=3.5)
    ax.fill_between(a, a, e, color=colour, alpha=0.13)
    ax.annotate(f"{lab}\nmax dev {res['tarp_max_dev']:.3f}", xy=(a[-6], e[-6]),
                xytext=(-6, 10 if res is tarp else -26), textcoords="offset points",
                ha="right", fontsize=8, color=colour)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
finish(ax, "TARP expected coverage (perturbative-Lund EMD)",
       r"credibility level $\alpha$", r"expected coverage ECP($\alpha$)")
plt.show()

for res, lab in ((tarp, "trained"), (tarp_sharp, "control (T=0.05)")):
    d68 = res["ecp_at"]["0.68"] - 0.68
    verdict = ("over-confident" if d68 < -0.03 else "over-dispersed" if d68 > 0.03
               else "consistent with calibrated")
    print(f"  {lab:<20} ECP(0.68) = {res['ecp_at']['0.68']:.3f}  "
          f"ECP(0.90) = {res['ecp_at']['0.90']:.3f}   -> {verdict}")

`ECP(0.68)` reads directly as *"at 68% credibility the posterior actually covered X% of the
time"* — the form worth quoting in a paper. The control curve sits **below** the diagonal
because a temperature-sharpened posterior concentrates on a handful of trees, so the truth
falls outside its credible regions far more often than the stated level admits.

Cost is `n_jets × (K+1)` EMD solves, and it needs the `[mbr]` extra (`pot`).

---
## 8. WP1 — is `log_prob` even a density?

The whole repo rests on one contract: `log_prob` returns `log q_φ(y|x)`, a normalized
density. Model selection by NLL and the likelihood-ratio deliverable both depend on it.

`models/diffusion.py` did not honour it. Its coordinate term is the denoising
score-matching **regression residual** used as a proxy — not the ELBO, not the
probability-flow-ODE likelihood — so it carries an unknown, context-dependent offset. Its
"NLL" is a relative score within that family and nothing more.

WP1 does two things about it. First, `exact_likelihood` becomes a class attribute, so
consumers warn on the *flag* rather than hard-coding a family name. Second, `model=cfm`
adds the exact-likelihood member of the same continuous-time family: conditional flow
matching (Lipman et al., arXiv:2210.02747) with the density obtained by integrating the
probability-flow ODE. The coordinate space is 4-dimensional, so the divergence is computed
**exactly** with 4 vector-Jacobian products per step — no Hutchinson estimator, hence a
deterministic likelihood rather than a stochastic one.

In [ ]:
def quick(sel, tag, epochs=6):
    cfg = load_config(sel + ["encoder=gru", "data=rntuple", f"data.path={JETS_ROOT}",
                             f"trainer.max_epochs={epochs}", "trainer.batch_size=128"])
    return train(cfg, tag)

m_cfm, nll_cfm, _, _ = quick(["model=cfm", "model.n_ode_steps=16", "model.ode_solver=heun"], "cfm")
m_cinn, nll_cinn, _, _ = quick(["model=cinn"], "cinn")
m_diff, nll_diff, _, _ = quick(["model=diffusion"], "diffusion")

print(f"\n{'family':<12}{'exact?':<9}{'val NLL/jet':>12}   comparable with the others?")
for name, m, v in (("ar_junipr_v3", model, nll_v3), ("cinn", m_cinn, nll_cinn),
                   ("cfm", m_cfm, nll_cfm), ("diffusion", m_diff, nll_diff)):
    ok = m.exact_likelihood
    print(f"{name:<12}{str(ok):<9}{v:>12.3f}   {'yes' if ok else 'NO - surrogate, unknown offset'}")

Three of those four numbers can be ranked against each other. The fourth cannot, and no
amount of staring at the table reveals which — which is precisely why the flag exists and
why `train` / `eval` / `serve` each print a warning when they report one.

`cfm` also illustrates why WP1 needed a second, smaller contract addition. Flow matching
*trains* by regressing a vector field and only integrates the ODE at evaluation. The
alternatives were to backpropagate through the ODE (slow, unstable, and it discards the
entire point of flow matching) or to let `log_prob` return the cheap surrogate during
training — which is the dishonesty WP1 exists to remove. So `training_objective` became a
hook that defaults to `-log_prob`; every other family's loop is bit-identical, and `cfm`
keeps an exact `log_prob`.

That is why its `train_nll` and `val_nll` are on different scales, and why `train` says so
at startup rather than leaving it to be discovered.

### The claim, checked numerically

"Normalized on the physical support" is falsifiable, so let us falsify it — Monte-Carlo
integrate `exp(log p)` over the physical box `(±half_u, ±half_v) × ℝ × (-π, π)` for a
frozen context, with a proposal built **independently** of the model's own bijections.

In [ ]:
def mc_normalization(m, cell=42, n_mc=40000, sp=1.8, sz=2.5, seed=1, sign=+1.0):
    """Importance-sampling estimate of the 4-d coordinate integral. `sign=-1` negates the
    divergence term — the control that shows the test can actually fail."""
    hu, hv = geom.half_u, geom.half_v
    cx, cy = geom.cell_center(cell)
    md_ = m.double()
    yc = torch.tensor([cell]).expand(n_mc)
    torch.manual_seed(0)
    ctx = torch.cat([torch.randn(1, md_.ctx_dim, dtype=torch.float64),
                     md_.cell_emb(torch.tensor([cell]))], -1).expand(n_mc, -1)
    g = torch.Generator().manual_seed(seed)
    s = torch.randn(n_mc, 4, generator=g, dtype=torch.float64)
    du, dv = hu * torch.tanh(sp * s[:, 0]), hv * torch.tanh(sp * s[:, 1])
    lnz, ps = sz * s[:, 2], math.pi * torch.tanh(sp * s[:, 3])
    def log_q(x, h):
        t = (x / h).clamp(-1 + 1e-14, 1 - 1e-14); z = torch.atanh(t) / sp
        return (-0.5 * z**2 - math.log(sp) - 0.5 * math.log(2 * math.pi)
                - math.log(h) - torch.log1p(-t * t))
    lq = (log_q(du, hu) + log_q(dv, hv) + log_q(ps, math.pi)
          + (-0.5 * (lnz / sz) ** 2 - math.log(sz) - 0.5 * math.log(2 * math.pi)))
    yraw = torch.stack([cx + du, cy + dv, lnz, ps], -1)
    s1, ldj = md_._to_std(yraw, yc)
    s0, acc = md_._ode(s1, ctx, reverse=True, with_divergence=True)
    lp = (-0.5 * s0**2 - 0.5 * math.log(2 * math.pi)).sum(-1) - sign * acc + ldj
    w = (lp - lq).exp()
    m.float()
    return float(w.mean()), float(w.std() / math.sqrt(n_mc))

est, se = mc_normalization(m_cfm.cpu())
bad, _ = mc_normalization(m_cfm.cpu(), sign=-1.0)
print(f"integral of exp(log p) over the physical support = {est:.4f} +/- {se:.4f}   (target 1)")
print(f"same integral with the divergence term negated    = {bad:.4f}"
      f"   ({abs(bad - 1) / se:.0f} sigma away — the test is not a tautology)")
m_cfm.to(device)

This is the test a discretized grid head could never pass, and it is not decorative: it
caught a genuine sign error in the reverse-pass divergence accounting while WP1 was being
written.

---
## 9. WP3 — the fixed-length bottleneck

`Encoder.forward` returns one pooled `ctx_dim` vector, and the decoder tiles it at every
step. So every hadron-level node reaches the parton-level decoder through a single 64-d
vector — and LundNet's graph structure is flattened before the decoder ever sees it.

`ar_junipr_v4` lets the decoder additionally **cross-attend** to the encoder's per-node
states. The attention is applied as a *residual*, so no head's input width changes and the
off path stays byte-identical.

Compare at **matched parameter count** — the attention adds ~25k parameters at
`dec_dim=64`, so a naive comparison measures capacity, not architecture. Shrinking
`dec_dim` to 52 brings v4 within 1.1% of v3.

In [ ]:
cfg_v4 = load_config([
    "model=ar_junipr_v4", "model.dec_dim=52", "encoder=gru",
    "data=rntuple", f"data.path={JETS_ROOT}",
    f"trainer.max_epochs={EPOCHS}", "trainer.batch_size=128",
])
model_v4, nll_v4, _, _ = train(cfg_v4, "ar_junipr_v4")

pits_v4 = coordinate_pits(model_v4, val_ds, geom, device, n_jets=N_EVAL, verbose=False)
tarp_v4 = run_tarp(model_v4, val_ds, geom, device, K=K_DRAWS, n_jets=N_EVAL,
                   n_refs=100, seed=0, verbose=False)

In [ ]:
n_v3 = sum(p.numel() for p in model.parameters())
n_v4 = sum(p.numel() for p in model_v4.parameters())

fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.2))
pairs = [("val NLL / jet", nll_v3, nll_v4, "lower is better"),
         ("max PIT KS", pits["ks_max"], pits_v4["ks_max"], "lower is better"),
         ("TARP max deviation", tarp["tarp_max_dev"], tarp_v4["tarp_max_dev"], "lower is better")]
for ax, (title, a_, b_, note) in zip(axes, pairs):
    bars = ax.bar([0, 1], [a_, b_], width=0.55, color=[C_BLUE, C_ORANGE], zorder=3)
    for bar, val in zip(bars, (a_, b_)):
        ax.annotate(f"{val:.3f}", (bar.get_x() + bar.get_width() / 2, val), xytext=(0, 4),
                    textcoords="offset points", ha="center", fontsize=8.5, color=INK)
    ax.set_xticks([0, 1]); ax.set_xticklabels([f"v3\n{n_v3/1e3:.0f}k", f"v4\n{n_v4/1e3:.0f}k"])
    ax.set_ylim(0, max(a_, b_) * 1.3)
    finish(ax, f"{title}\n({note})")
fig.suptitle("matched parameter count: pooled context (v3) vs cross-attention (v4)",
             y=1.06, fontsize=10)
plt.show()

print(f"parameters: v3 {n_v3:,}   v4 {n_v4:,}   ({100*(n_v4-n_v3)/n_v3:+.1f}%)")
print(f"val NLL/jet: {nll_v3:.3f} -> {nll_v4:.3f}   ({nll_v4-nll_v3:+.3f} nats)")

A likelihood improvement at matched capacity is necessary but not sufficient — a model can
buy NLL while getting *less* calibrated. That is why the PIT and TARP panels sit beside it,
and why adoption for physics runs goes through the full A/B
([`scripts/ab_v2_v3.py`](../scripts/ab_v2_v3.py)) rather than a single number.

---
## 10. Summary

What this notebook demonstrated on 54 007 real PYTHIA jets:

In [ ]:
rows = [
    ("SBC-on-N (old suite)", f"chi2 {base_metrics['sbc_chi2_uniform']:.1f}, "
                             f"rank mean {base_metrics['sbc_rank_mean']:.3f}",
     "near-tautological for v3 — cannot referee the A/B"),
    ("per-coordinate PIT", f"max KS {pits['ks_max']:.4f} over {len(names)} coordinates",
     "the kinematics; U-shape = over-confident"),
    ("region stratification", f"{len([r for r in REGION_LABELS if r in by_region])} Lund quadrants",
     "whether calibration holds locally, not on average"),
    ("TARP", f"max dev {tarp['tarp_max_dev']:.3f}, ECP(0.68) = {tarp['ecp_at']['0.68']:.3f}",
     "the joint tree, in the perturbative-Lund EMD"),
    ("exact_likelihood", "cfm/cinn/ar_* True, diffusion False",
     "which NLLs may be compared at all"),
    ("support guard", f"P(N > 25) = {ok['tail_fraction']:.1e}",
     "a categorical q(N|x) silently mis-normalizes past its support"),
    ("cross-attention", f"val NLL {nll_v3:.3f} -> {nll_v4:.3f} at matched params",
     "the pooled-context bottleneck"),
]
w = max(len(r[0]) for r in rows)
for name, result, why in rows:
    print(f"  {name:<{w}}  {result:<44}  {why}")

### Reproducing this outside the notebook

```bash
# the whole suite on any checkpoint
h2p-rsd-junipr eval runs/<id>/best.ckpt \
    experiment.pit_coords=true experiment.stratify_regions=true experiment.tarp=true

# the v2-vs-v3 A/B the suite gates (each arm trained once, every decode cell evaluated)
python scripts/ab_v2_v3.py --preset presets/ab_v2_v3.yaml --out runs/ab_v2_v3
```

`eval` writes `eval_metrics.json` and the three calibration figures beside the checkpoint.

### Where to read further

- [`docs/CONFIGURATION.md` §8](../docs/CONFIGURATION.md) — every switch, and how to read each metric
- [`docs/CONFIGURATION.md` §7](../docs/CONFIGURATION.md) — which decode knobs are still live under v3
- [`docs/PLAN_UPDATES.md`](../docs/PLAN_UPDATES.md) — the work packages, the deviations, the exit criteria
- [`docs/README_PHYSICS.md`](../docs/README_PHYSICS.md) — the physics the posterior is modelling